# Personal Assistant Using Sub-Agent Architecture
### Official LangChain Pattern: Supervisor Hub-and-Spoke with Specialized Calendar & Email Subagents

This notebook implements the official LangChain tutorial [Build a Personal Assistant with Subagents](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents-personal-assistant).

---

### Architectural Concept

```
                          ┌────────────────────────────────┐
                          │          User Request          │
                          └───────────────┬────────────────┘
                                          │
                                          ▼
                          ┌────────────────────────────────┐
                          │   Main Personal Assistant      │
                          │      (Supervisor Agent)        │
                          └───────┬────────────────┬───────┘
                                  │                │
            call_calendar_agent() │ (as a tool)    │ call_email_agent() (as a tool)
                                  ▼                ▼
                     ┌──────────────────┐    ┌──────────────────┐
                     │  Calendar Agent  │    │   Email Agent    │
                     │  (Subagent A)    │    │   (Subagent B)   │
                     └────────┬─────────┘    └────────┬─────────┘
                              │                       │
                     ┌────────┴─────────┐    ┌────────┴─────────┐
                     │   Calendar DB    │    │     Email DB     │
                     │  (Events/Slots)  │    │ (Inbox & Drafts) │
                     └──────────────────┘    └──────────────────┘
```

### Why Use the Subagents Pattern?
1. **Context Isolation**: Subagents run in isolated scratchpads. The Main Agent's conversation history doesn't get cluttered with internal calendar API calls or email search iterations.
2. **Specialized Prompts**: The Calendar Agent has dedicated scheduling logic; the Email Agent has communication formatting rules.
3. **Tool Partitioning**: Instead of one monolithic agent with 15+ tools (which confuses LLMs), each agent only receives 3-4 domain-specific tools.
4. **Single User Interface**: The user only interacts with the Main Assistant, who orchestrates the team seamlessly.


## 1. Setup & Environment Variables
Configure Google Gemini API keys and environment variables.


In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

# Load API keys from root .env if present
if os.path.exists(".env"):
    load_dotenv(dotenv_path=".env")
elif os.path.exists("../.env"):
    load_dotenv(dotenv_path="../.env")
else:
    load_dotenv()

if "GEMINI_API_KEY" not in os.environ and "GOOGLE_API_KEY" in os.environ:
    os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"]

print("✓ Environment configured successfully!")
print(f"Google API Key present: {'GEMINI_API_KEY' in os.environ or 'GOOGLE_API_KEY' in os.environ}")


✓ Environment configured successfully!
Google API Key present: True


## 2. Mock Databases (Calendar & Email)
To make this assistant realistic and testable, we set up simulated in-memory databases for calendar schedules and emails.


In [2]:
# Simulated Calendar Database
CALENDAR_DB = {
    "Monday": [
        {"title": "Weekly Team Standup", "time": "10:00 AM", "duration": "30 mins", "with": "Engineering Team"},
        {"title": "1-on-1 with Manager", "time": "02:00 PM", "duration": "45 mins", "with": "Alex Rivera"}
    ],
    "Tuesday": [
        {"title": "Product Design Review", "time": "11:00 AM", "duration": "60 mins", "with": "Design Team"},
        {"title": "Dentist Appointment", "time": "04:30 PM", "duration": "45 mins", "with": "Dr. Davis"}
    ],
    "Wednesday": [
        {"title": "Quarterly Planning", "time": "09:00 AM", "duration": "90 mins", "with": "Executive Team"}
    ],
    "Thursday": [],
    "Friday": [
        {"title": "Sprint Retrospective", "time": "03:00 PM", "duration": "60 mins", "with": "Engineering Team"}
    ]
}

# Simulated Email Directory & Database
CONTACTS_DIR = {
    "alice": "alice.smith@enterprise.com",
    "bob": "bob.jones@consulting.org",
    "sarah": "sarah.connor@techstart.io",
    "david": "david.kim@partners.net"
}

EMAIL_DB = {
    "inbox": [
        {"from": "sarah.connor@techstart.io", "subject": "Project Sync this week?", "date": "Yesterday", "snippet": "Hey, do you have 30 minutes tomorrow afternoon to discuss the Q3 roadmap?"},
        {"from": "alex.rivera@enterprise.com", "subject": "1-on-1 Agenda", "date": "2 days ago", "snippet": "Looking forward to our sync on Monday at 2 PM."}
    ],
    "sent_drafts": []
}

print("✓ Calendar & Email databases initialized.")


✓ Calendar & Email databases initialized.


## 3. Define Domain Tools
We define concrete tools for:
- **Calendar**: `get_calendar_events`, `check_availability`, `schedule_event`.
- **Email**: `search_inbox`, `draft_email`, `send_email`.


In [3]:
from langchain.tools import tool
from typing import List

# ==================== CALENDAR TOOLS ====================

@tool
def get_calendar_events(day: str) -> str:
    """Retrieve all scheduled events and meetings for a given day (e.g. 'Monday', 'Tuesday')."""
    day_key = day.capitalize()
    events = CALENDAR_DB.get(day_key, None)
    if events is None:
        return f"Unknown day '{day}'. Please specify a valid day of the week."
    if not events:
        return f"Your schedule is completely clear on {day_key}. No meetings scheduled!"
    
    output = f"Scheduled events for {day_key}:\n"
    for e in events:
        output += f"- [{e['time']}] {e['title']} ({e['duration']}) with {e['with']}\n"
    return output

@tool
def check_availability(day: str, time_slot: str) -> str:
    """Check if a specific time slot on a given day is free or conflicts with existing meetings."""
    day_key = day.capitalize()
    events = CALENDAR_DB.get(day_key, [])
    
    # Check for direct conflicts
    for e in events:
        if e['time'].lower() == time_slot.lower():
            return f"CONFLICT DETECTED: You already have '{e['title']}' scheduled at {time_slot} on {day_key}."
            
    return f"AVAILABLE: The time slot {time_slot} on {day_key} is completely open and free!"

@tool
def schedule_event(title: str, day: str, time_slot: str, participants: str) -> str:
    """Book a new meeting into your calendar schedule."""
    day_key = day.capitalize()
    if day_key not in CALENDAR_DB:
        CALENDAR_DB[day_key] = []
        
    # Check conflict first
    for e in CALENDAR_DB[day_key]:
        if e['time'].lower() == time_slot.lower():
            return f"ERROR: Cannot schedule '{title}'. Conflict with '{e['title']}' at {time_slot}."
            
    new_event = {
        "title": title,
        "time": time_slot,
        "duration": "30 mins",
        "with": participants
    }
    CALENDAR_DB[day_key].append(new_event)
    return f"SUCCESS: Meeting '{title}' scheduled for {day_key} at {time_slot} with {participants}."


# ==================== EMAIL TOOLS ====================

@tool
def search_inbox(query: str) -> str:
    """Search past emails and messages by sender name, subject, or keywords."""
    results = []
    q_lower = query.lower()
    for mail in EMAIL_DB["inbox"]:
        if q_lower in mail["from"].lower() or q_lower in mail["subject"].lower() or q_lower in mail["snippet"].lower():
            results.append(f"From: {mail['from']} | Subject: {mail['subject']} | Snippet: {mail['snippet']}")
            
    if not results:
        return f"No emails found matching query: '{query}'."
    return "Inbox Search Results:\n" + "\n".join(results)

@tool
def draft_email(recipient: str, subject: str, body: str) -> str:
    """Create and save a new email draft to the outbox for review."""
    # Resolve contact email if name was passed
    resolved_to = CONTACTS_DIR.get(recipient.lower(), recipient)
    draft = {
        "to": resolved_to,
        "subject": subject,
        "body": body,
        "status": "Draft Saved"
    }
    EMAIL_DB["sent_drafts"].append(draft)
    return (
        f"EMAIL DRAFT CREATED:\n"
        f"- To: {resolved_to}\n"
        f"- Subject: {subject}\n"
        f"- Body: {body}\n"
        f"Draft status: Ready for user review."
    )

@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """Send an email directly to the recipient."""
    resolved_to = CONTACTS_DIR.get(recipient.lower(), recipient)
    record = {
        "to": resolved_to,
        "subject": subject,
        "body": body,
        "status": "SENT"
    }
    EMAIL_DB["sent_drafts"].append(record)
    return f"SUCCESS: Email sent to {resolved_to} with subject '{subject}'."

print("✓ Calendar & Email tools defined.")


✓ Calendar & Email tools defined.


## 4. Build Specialized Subagents (`calendar_agent` & `email_agent`)
Each subagent is instantiated using LangChain's `create_agent` with its dedicated tools and system prompt.


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)

# 1. Specialized Calendar Subagent
CALENDAR_SYSTEM_PROMPT = """You are an Expert Calendar & Scheduling Assistant.
Your sole job is to inspect calendar events, verify availability, and schedule meetings.

Rules:
1. Always check availability before booking an event.
2. If there is a conflict, clearly report what meeting conflicts and suggest checking other times.
3. Be concise, precise, and professional."""

calendar_subagent = create_agent(
    model=llm,
    tools=[get_calendar_events, check_availability, schedule_event],
    system_prompt=CALENDAR_SYSTEM_PROMPT
)

# 2. Specialized Email Subagent
EMAIL_SYSTEM_PROMPT = """You are an Expert Email & Communications Assistant.
Your sole job is to search the inbox, find sender details, draft clear emails, and handle messages.

Rules:
1. When drafting an email, keep it courteous, professional, and clear.
2. Resolve contact names (like Sarah, Bob, Alice) to proper email addresses using your tools.
3. Report the drafted or sent email details clearly."""

email_subagent = create_agent(
    model=llm,
    tools=[search_inbox, draft_email, send_email],
    system_prompt=EMAIL_SYSTEM_PROMPT
)

print("✓ Calendar Subagent and Email Subagent instantiated.")


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✓ Calendar Subagent and Email Subagent instantiated.


## 5. Expose Subagents as Tools for the Supervisor
Here is the core mechanism of the **Subagent Pattern**:
The Main Personal Assistant does **not** call calendar or email APIs directly. Instead, it delegates to the subagents by invoking them as **tools**!


In [5]:
@tool
def call_calendar_agent(instruction: str) -> str:
    """Delegates calendar and scheduling tasks to the specialized Calendar Subagent.
    Use this for: checking availability, viewing daily agenda, or booking calendar events."""
    response = calendar_subagent.invoke({
        "messages": [{"role": "user", "content": instruction}]
    })
    # Extract final text output from the subagent
    last_msg = response["messages"][-1]
    content = last_msg.content[0]["text"] if isinstance(last_msg.content, list) else last_msg.content
    return content

@tool
def call_email_agent(instruction: str) -> str:
    """Delegates email and communications tasks to the specialized Email Subagent.
    Use this for: searching inbox messages, drafting emails, or sending messages to contacts."""
    response = email_subagent.invoke({
        "messages": [{"role": "user", "content": instruction}]
    })
    # Extract final text output from the subagent
    last_msg = response["messages"][-1]
    content = last_msg.content[0]["text"] if isinstance(last_msg.content, list) else last_msg.content
    return content

print("✓ Delegation tools (call_calendar_agent, call_email_agent) defined.")


✓ Delegation tools (call_calendar_agent, call_email_agent) defined.


## 6. Main Personal Assistant (Supervisor)
The Main Assistant acts as the user's primary executive assistant. It coordinates between calendar and email subagents and formats a unified, helpful response.


In [6]:
SUPERVISOR_SYSTEM_PROMPT = """You are an Executive Personal Assistant.
You help your busy executive manage their time, calendar, and email communications.

You have access to two specialized subagents:
1. call_calendar_agent: for all scheduling, availability checks, and meeting bookings.
2. call_email_agent: for searching inbox messages, drafting emails, and sending messages.

Workflow Guidelines:
- If a user asks to schedule a meeting with someone and email them, coordinate both:
  1. Call call_calendar_agent to verify the slot and book the event.
  2. Call call_email_agent to draft or send the confirmation email with the meeting details.
- Synthesize the results into a single, cohesive, polite response.
- Do not make up facts; rely on your specialized subagents."""

personal_assistant = create_agent(
    model=llm,
    tools=[call_calendar_agent, call_email_agent],
    system_prompt=SUPERVISOR_SYSTEM_PROMPT
)

print("✓ Main Personal Assistant Supervisor Agent instantiated.")


✓ Main Personal Assistant Supervisor Agent instantiated.


## 7. Execution Runs: Testing Real-World Scenarios
We test two representative personal assistant scenarios:
1. **Cross-Domain Multi-Task**: Checking schedule, booking a meeting with Sarah, and drafting a confirmation email.
2. **Conflict Detection & Handling**: Trying to book a time slot that is already occupied.


In [7]:
from rich.console import Console
from rich.panel import Panel

console = Console()

def run_assistant(query: str):
    console.print(f"\n[bold yellow]User Request:[/bold yellow] [bold white]{query}[/bold white]")
    console.print("─"*80, style="dim")
    
    response = personal_assistant.invoke({
        "messages": [{"role": "user", "content": query}]
    })
    
    last_msg = response["messages"][-1]
    final_text = last_msg.content[0]["text"] if isinstance(last_msg.content, list) else last_msg.content
    
    console.print(Panel(final_text, title="[bold green]Personal Assistant Response[/bold green]", border_style="green"))
    return final_text

# Scenario 1: Cross-Domain Multi-Task (Calendar Booking + Email Drafting)
scenario_1 = (
    "Can you check if I'm free tomorrow (Tuesday) at 2:00 PM? "
    "If I am, please schedule a 30-minute 'Q3 Product Roadmap Sync' with Sarah, "
    "and draft a confirmation email to her."
)

output_1 = run_assistant(scenario_1)


User Request: Can you check if I'm free tomorrow (Tuesday) at 2:00 PM? If I am, please schedule a 30-minute 'Q3 
Product Roadmap Sync' with Sarah, and draft a confirmation email to her.

────────────────────────────────────────────────────────────────────────────────

╭────────────────────────────────────────── Personal Assistant Response ──────────────────────────────────────────╮
│ You are available tomorrow, Tuesday, at 2:00 PM. The 'Q3 Product Roadmap Sync' with Sarah has been successfully │
│ scheduled for Tuesday at 2:00 PM for 30 minutes.                                                                │
│                                                                                                                 │
│ A confirmation email has been drafted to Sarah with the subject "Q3 Product Roadmap Sync - Meeting              │
│ Confirmation". The email confirms the meeting for tomorrow, Tuesday, at 2:00 PM for 30 minutes.                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
output_1

'You are available tomorrow, Tuesday, at 2:00 PM. The \'Q3 Product Roadmap Sync\' with Sarah has been successfully scheduled for Tuesday at 2:00 PM for 30 minutes.\n\nA confirmation email has been drafted to Sarah with the subject "Q3 Product Roadmap Sync - Meeting Confirmation". The email confirms the meeting for tomorrow, Tuesday, at 2:00 PM for 30 minutes.'

### Scenario 2: Scheduling Conflict Detection
We request a meeting on Monday at 10:00 AM, which conflicts with our existing Weekly Standup.


In [10]:
# Scenario 2: Attempting to schedule during an existing conflict
scenario_2 = (
    "Please schedule a 1-hour Strategy Meeting with David on Monday at 10:00 AM."
)

output_2 = run_assistant(scenario_2)


User Request: Please schedule a 1-hour Strategy Meeting with David on Monday at 10:00 AM.

────────────────────────────────────────────────────────────────────────────────

╭────────────────────────────────────────── Personal Assistant Response ──────────────────────────────────────────╮
│ I'm sorry, but I cannot schedule the Strategy Meeting for Monday at 10:00 AM. You have a 'Weekly Team Standup'  │
│ at that time. Please suggest an alternative time.                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯